# Final Inference Notebook — Offline Kaggle Submission

This notebook loads both the base model and the trained LoRA adapter from attached Kaggle datasets.  
It does **not** download the model from Hugging Face and does **not** rely on hardcoded usernames.

Required Kaggle inputs:
1. competition dataset containing `test.csv` and images
2. offline base model dataset containing the full `HuggingFaceTB/SmolVLM-500M-Instruct` snapshot
3. final adapter dataset containing `adapter_config.json` and `adapter_model.safetensors`

Output:
- `/kaggle/working/submission.csv`
- `/kaggle/working/inference_manifest.json`
- optional validation sanity metrics if `val.csv` is available


In [ ]:
# 0. Imports only. No pip install here because final Kaggle submission should run offline.

import os
import sys
import glob
import json
import time
import random
import hashlib
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 1. Config.

CONFIG = {
    "seed": 42,
    "img_size": 224,
    "max_context_chars": 400,
    "output_path": "/kaggle/working/submission.csv" if Path("/kaggle/working").exists() else "submission.csv",

    # Keep quick validation on by default if val.csv is attached.
    # This is a sanity metric, not a clean ablation if your adapter was trained on train+val.
    "run_val_sanity": True,
    "val_sanity_samples": 64,

    # Full validation can be slow. Use RUN_FULL_VAL=1 if needed.
    "run_full_val": os.environ.get("RUN_FULL_VAL", "0") == "1",
}

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])
print(json.dumps(CONFIG, indent=2))


In [ ]:
# 2. Auto-detect Kaggle/local data, base model, and adapter paths.

def _is_kaggle() -> bool:
    return Path("/kaggle/input").exists()

def print_inputs(max_lines: int = 100):
    root = Path("/kaggle/input")
    if not root.exists():
        return
    print("Mounted /kaggle/input paths:")
    shown = 0
    for p in sorted(root.rglob("*")):
        print(" ", p)
        shown += 1
        if shown >= max_lines:
            print(f"  ... shown first {max_lines} paths only")
            break

def find_data_dir() -> Path:
    env = os.environ.get("DATA_DIR")
    if env:
        p = Path(env)
        if not (p / "test.csv").exists():
            raise FileNotFoundError(f"DATA_DIR={p} does not contain test.csv")
        return p

    if _is_kaggle():
        matches = sorted(Path("/kaggle/input").rglob("test.csv"))
        if not matches:
            print_inputs()
            raise FileNotFoundError("Could not find test.csv. Attach the competition dataset.")
        # Prefer folder with train.csv too, but test.csv is enough for inference.
        matches = sorted(matches, key=lambda x: 0 if (x.parent / "train.csv").exists() else 1)
        return matches[0].parent

    p = Path("data")
    if not (p / "test.csv").exists():
        raise FileNotFoundError("Local data/test.csv not found. Put files under ./data or set DATA_DIR.")
    return p

def looks_like_adapter_dir(p: Path) -> bool:
    return (
        (p / "adapter_config.json").exists()
        and ((p / "adapter_model.safetensors").exists() or (p / "adapter_model.bin").exists())
    )

def find_adapter_dir() -> Path:
    env = os.environ.get("ADAPTER_DIR")
    if env:
        p = Path(env)
        if not looks_like_adapter_dir(p):
            raise FileNotFoundError(f"ADAPTER_DIR={p} does not look like a PEFT adapter directory.")
        return p

    search_roots = [Path("/kaggle/input")] if _is_kaggle() else [Path("outputs"), Path(".")]
    candidates = []
    for root in search_roots:
        if root.exists():
            for cfg in root.rglob("adapter_config.json"):
                parent = cfg.parent
                if looks_like_adapter_dir(parent):
                    candidates.append(parent)

    candidates = sorted(
        candidates,
        key=lambda p: (0 if "final" in str(p).lower() or "adapter" in str(p).lower() else 1, len(str(p)))
    )
    if not candidates:
        print_inputs()
        raise FileNotFoundError(
            "Could not find adapter_config.json + adapter_model.safetensors. "
            "Attach the final adapter Kaggle dataset or set ADAPTER_DIR."
        )
    return candidates[0]

def looks_like_base_model_dir(p: Path) -> bool:
    has_config = (p / "config.json").exists()
    has_model_weights = (
        (p / "model.safetensors").exists()
        or (p / "model.safetensors.index.json").exists()
        or (p / "pytorch_model.bin").exists()
        or (p / "pytorch_model.bin.index.json").exists()
    )
    is_adapter = (p / "adapter_config.json").exists()
    return has_config and has_model_weights and not is_adapter

def find_base_model_dir(adapter_dir: Path) -> Path:
    """
    Strict offline source. We intentionally do not fall back to Hugging Face model ID here.
    """
    env = os.environ.get("BASE_MODEL_DIR")
    if env:
        p = Path(env)
        if not looks_like_base_model_dir(p):
            raise FileNotFoundError(f"BASE_MODEL_DIR={p} does not look like a full base-model snapshot.")
        return p

    # If the adapter bundle contains a nested base_model folder, use it.
    nested = adapter_dir / "base_model"
    if looks_like_base_model_dir(nested):
        return nested

    if _is_kaggle():
        candidates = []
        for cfg in Path("/kaggle/input").rglob("config.json"):
            parent = cfg.parent
            if looks_like_base_model_dir(parent):
                candidates.append(parent)

        candidates = sorted(candidates, key=lambda p: (0 if "smol" in str(p).lower() else 1, len(str(p))))
        if candidates:
            return candidates[0]

    print_inputs()
    raise FileNotFoundError(
        "Could not find offline base model snapshot. Attach a Kaggle dataset with the full "
        "HuggingFaceTB/SmolVLM-500M-Instruct files or set BASE_MODEL_DIR. "
        "Do not rely on online from_pretrained() for final submission."
    )

DATA_DIR = find_data_dir()
ADAPTER_DIR = find_adapter_dir()
BASE_MODEL_DIR = find_base_model_dir(ADAPTER_DIR)

print("DATA_DIR:", DATA_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("BASE_MODEL_DIR:", BASE_MODEL_DIR)


In [ ]:
# 3. Load CSVs.

def parse_choices(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        return json.loads(x)
    raise TypeError(f"Unsupported choices value: {type(x)}")

def load_split(name: str, required: bool) -> pd.DataFrame | None:
    path = DATA_DIR / f"{name}.csv"
    if not path.exists():
        if required:
            raise FileNotFoundError(path)
        return None
    df = pd.read_csv(path)
    if "choices" in df.columns:
        df["choices"] = df["choices"].apply(parse_choices)
        df["num_choices"] = df["choices"].apply(len)
    return df

test_df = load_split("test", required=True)
val_df = load_split("val", required=False)

print("test rows:", len(test_df))
print("val rows:", 0 if val_df is None else len(val_df))
display(test_df.head(2))


In [ ]:
# 4. Prompt and image functions. Must match training.

CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []

    lecture = row.get("lecture", "")
    hint = row.get("hint", "")

    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip()[:CONFIG["max_context_chars"]])
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip()[:CONFIG["max_context_chars"]])

    context_str = "\n".join(context_parts)
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(row["choices"])
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"

    return prompt

def _resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    rel = Path(str(rel_path))
    parts = rel.parts

    candidates = [
        data_dir / rel,
        data_dir / "images" / rel,
        data_dir / rel.name,
        data_dir / "images" / rel.name,
    ]
    if len(parts) > 1 and parts[0] == "images":
        candidates.append(data_dir / Path(*parts[1:]))
        candidates.append(data_dir / "images" / Path(*parts[1:]))

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError("Image not found. Tried:\n" + "\n".join(f"  {p}" for p in candidates))

def load_image(rel_path: str) -> Image.Image:
    return Image.open(_resolve_image_path(DATA_DIR, rel_path)).convert("RGB").resize(
        (CONFIG["img_size"], CONFIG["img_size"]),
        Image.BICUBIC,
    )

print(build_prompt(test_df.iloc[0], include_answer=False))
print("Image sanity:", load_image(test_df.iloc[0]["image_path"]).size)


In [ ]:
# 5. Load processor, base model, and LoRA adapter from local paths only.

from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# Prefer processor saved with the adapter; otherwise use base snapshot processor.
PROCESSOR_DIR = ADAPTER_DIR if (ADAPTER_DIR / "preprocessor_config.json").exists() else BASE_MODEL_DIR

processor = AutoProcessor.from_pretrained(PROCESSOR_DIR, local_files_only=True)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    local_files_only=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=True,
)
model.eval()

print("Processor loaded from:", PROCESSOR_DIR)
print("Base model loaded from:", BASE_MODEL_DIR)
print("Adapter loaded from:", ADAPTER_DIR)
print("Model ready.")


In [ ]:
# 6. Prediction/evaluation functions.

LETTER_TOKEN_IDS = {
    letter: processor.tokenizer.encode(f" {letter}", add_special_tokens=False)[-1]
    for letter in CHOICE_LETTERS
}

def get_input_device(model):
    if hasattr(model, "hf_device_map"):
        for dev in model.hf_device_map.values():
            if dev in ("cpu", "disk"):
                continue
            if isinstance(dev, int):
                return torch.device(f"cuda:{dev}" if torch.cuda.is_available() else "cpu")
            if isinstance(dev, str) and dev.isdigit():
                return torch.device(f"cuda:{dev}" if torch.cuda.is_available() else "cpu")
            return torch.device(dev)
    return next(model.parameters()).device

def move_inputs_to_device(inputs: dict, device: torch.device) -> dict:
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in inputs.items()}

def predict_loglikelihood(image: Image.Image, prompt_text: str, num_choices: int) -> int:
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt")
    inputs = move_inputs_to_device(inputs, get_input_device(model))

    with torch.inference_mode():
        logits = model(**inputs).logits

    last_logits = logits[0, -1, :]
    log_probs = F.log_softmax(last_logits, dim=-1)
    scores = [
        log_probs[LETTER_TOKEN_IDS[CHOICE_LETTERS[i]]].item()
        for i in range(num_choices)
    ]
    return int(np.argmax(scores))

def evaluate(df: pd.DataFrame, limit: int | None = None, output_prefix: str = "validation"):
    eval_df = df.copy().reset_index(drop=True)
    if limit is not None:
        eval_df = eval_df.head(limit)

    preds = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Evaluating {output_prefix}"):
        image = load_image(row["image_path"])
        prompt = build_prompt(row, include_answer=False)
        preds.append(predict_loglikelihood(image, prompt, int(row["num_choices"])))

    out = eval_df.copy()
    out["pred"] = preds
    out["correct"] = (out["pred"].astype(int) == out["answer"].astype(int)).astype(int)

    metrics = {
        "n": int(len(out)),
        "accuracy": float(out["correct"].mean()) if len(out) else None,
        "note": "Diagnostic only if the adapter was trained on train+val.",
        "breakdowns": {},
    }

    for col in ["num_choices", "subject", "topic", "category", "grade"]:
        if col in out.columns:
            metrics["breakdowns"][col] = (
                out.groupby(col)["correct"]
                .agg(["count", "mean"])
                .reset_index()
                .rename(columns={"mean": "accuracy"})
                .to_dict(orient="records")
            )

    out_path = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    out.to_csv(out_path / f"{output_prefix}_predictions.csv", index=False)
    with open(out_path / f"{output_prefix}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    print(json.dumps(metrics, indent=2)[:4000])
    return out, metrics

def make_submission(df: pd.DataFrame, output_path: str | Path):
    preds = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Predicting test"):
        image = load_image(row["image_path"])
        prompt = build_prompt(row, include_answer=False)
        preds.append(predict_loglikelihood(image, prompt, int(row["num_choices"])))

    submission = pd.DataFrame({"id": df["id"].values, "answer": preds})
    submission.to_csv(output_path, index=False)

    assert len(submission) == len(df), "Wrong number of rows."
    assert submission["answer"].notna().all(), "Missing predictions."
    assert submission["answer"].map(lambda x: isinstance(int(x), int)).all(), "Answers must be integer class indices."

    print(f"Saved {len(submission)} rows -> {output_path}")
    return submission


In [ ]:
# 7. Optional local validation sanity metric.

if val_df is not None and "answer" in val_df.columns and CONFIG["run_val_sanity"]:
    sanity_n = min(CONFIG["val_sanity_samples"], len(val_df))
    print(f"Running validation sanity check on {sanity_n} examples.")
    sanity_preds, sanity_metrics = evaluate(val_df, limit=sanity_n, output_prefix="validation_sanity")
else:
    print("Skipping validation sanity check.")


In [ ]:
# 8. Optional full validation metric.

if val_df is not None and "answer" in val_df.columns and CONFIG["run_full_val"]:
    full_preds, full_metrics = evaluate(val_df, limit=None, output_prefix="validation_full")
else:
    print("Skipping full validation. Set RUN_FULL_VAL=1 to enable.")


In [ ]:
# 9. Generate final Kaggle submission.

submission = make_submission(test_df, CONFIG["output_path"])
display(submission.head())


In [ ]:
# 10. Save inference manifest for reproducibility.

def sha256_if_small(path: Path, max_bytes: int = 20_000_000):
    if not path.exists() or not path.is_file() or path.stat().st_size > max_bytes:
        return None
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()

manifest = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "config": CONFIG,
    "data_dir": str(DATA_DIR),
    "base_model_dir": str(BASE_MODEL_DIR),
    "adapter_dir": str(ADAPTER_DIR),
    "processor_dir": str(PROCESSOR_DIR),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "adapter_files_sha256": {
        p.name: sha256_if_small(p)
        for p in sorted(ADAPTER_DIR.glob("*"))
        if p.is_file() and sha256_if_small(p) is not None
    },
    "submission_path": CONFIG["output_path"],
    "submission_rows": int(len(submission)),
}

manifest_path = Path("/kaggle/working/inference_manifest.json") if Path("/kaggle/working").exists() else Path("inference_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest ->", manifest_path)
print(json.dumps(manifest, indent=2)[:3000])
